# Семинар 5. Какой канал окупается

Легенда:
Маркетинг делит бюджет на следующий квартал и задаёт один вопрос: в какой канал
привлечения добавить денег, а какой резать. Каналов шесть.

Ответ — три числа на канал: сколько стоит привести клиента (**CAC**), сколько
он принесёт за всю жизнь (**LTV**) и за сколько месяцев окупится.

По дороге три развилки, и на каждой ответ можно незаметно испортить: чьи сутки
считать, с каким окном мерить воронку и на кого делить расходы.

Метки те же, что на третьем семинаре:

| Метка | Что значит |
|---|---|
| **понимать** | спросят на защите и на проверочной |
| **уметь писать** | понадобится в проекте |
| **знать, что существует** | достаточно помнить, что такое бывает, и загуглить |


## Выгрузка данных


In [ ]:
import numpy as np
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC — так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


SEGMENT = "region_code <= 20"   # двадцать регионов из восьмидесяти девяти, как на прошлых парах

print("подключились как:", q("select current_user as login").iloc[0, 0])


In [ ]:
clients = q(f"""
    select client_id, registered_at, time_zone, acquisition_channel
    from prime.clients
    where {SEGMENT}
    order by client_id
""")

# Канал у подписки — это канал её владельца.
subs = q(f"""
    select s.subscription_id, s.owner_client_id, s.started_at, s.ended_at,
           c.acquisition_channel
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
    where c.{SEGMENT}
    order by s.subscription_id
""")

print(f"клиентов: {len(clients):>8}")
print(f"подписок: {len(subs):>8}")


---

## 1. Чей июль

Все три числа мы будем считать «за июль». Но сначала — письмо из поддержки.

> Клиент из Владивостока: «**Первого августа** ночью у меня списали 449 рублей.
> Я ничего не продлевал, верните деньги».
>
> Оператор открывает платежи клиента за первое августа — и не находит ничего.


In [ ]:
# Платежи июля с запасом в один день с каждой стороны. Зачем запас — увидим
# через пять минут.
pay = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount,
           c.time_zone, c.acquisition_channel
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    join prime.clients c on c.client_id = s.owner_client_id
    where c.{SEGMENT}
      and p.status = 'success'
      and p.paid_at >= '2026-06-30' and p.paid_at < '2026-08-02'
    order by p.payment_id
""")

print(len(pay), "платежей")
print("тип колонки paid_at:", pay["paid_at"].dtype)


In [ ]:
# Тот самый платёж: клиент из Владивостока, вечер 31 июля по часам базы.
one = pay[(pay["time_zone"] == "Asia/Vladivostok")
          & (pay["paid_at"] >= "2026-07-31 14:00")
          & (pay["paid_at"] < "2026-08-01")].sort_values("payment_id").iloc[0]

print("в базе:", one["paid_at"])
print("сумма: ", one["amount"])


In [ ]:
# В базе платёж 31 июля. Клиент говорит — первого августа. Кто прав?
# Переведите момент платежа в пояс клиента.
print("у клиента:", one["paid_at"].tz_convert(ЗАПОЛНИТЕ))


**Правы оба.** Это один и тот же момент: в базе время лежит в UTC, а сутки клиента
сдвинуты на десять часов вперёд. Оператор искал платёж «за первое августа» по
часам базы — а по часам базы первое августа ещё не наступило.

Колонка `paid_at` — время **с поясом**: тип `datetime64[us, UTC]`. Это отдельный
тип, и с временем без пояса он не сравнивается.


In [ ]:
# Сколько платежей прошло с первого июля? Три способа написать границу.
try:
    print((pay["paid_at"] >= pd.Timestamp("2026-07-01")).sum())          # без пояса
except TypeError as e:
    print("TypeError:", e)

print("строкой:  ", (pay["paid_at"] >= "2026-07-01").sum())              # молча
print("с поясом: ", (pay["paid_at"] >= pd.Timestamp("2026-07-01", tz=ЗАПОЛНИТЕ)).sum())


**Понимать.** `Timestamp` без пояса падает громко — и это хорошо: pandas отказывается
гадать, в каком поясе вы написали дату. Строка проходит **молча**: pandas решает
за вас, что это UTC. Сегодня он угадал.


In [ ]:
def local_time(df: pd.DataFrame, moment: str = "paid_at", zone: str = "time_zone") -> pd.Series:
    """Время на часах клиента: у каждой строки свой пояс."""
    parts = []
    for tz, g in df.groupby(zone):
        # Переводим в пояс группы, потом снимаем пояс — остаётся время «на часах».
        # Чем снять пояс, не сдвинув время?
        parts.append(g[moment].dt.tz_convert(tz).dt.tz_localize(ЗАПОЛНИТЕ))
    return pd.concat(parts).sort_index()


pay["local"] = local_time(pay)

pay[["paid_at", "time_zone", "local"]].head()


In [ ]:
july = pay[(pay["paid_at"] >= "2026-07-01") & (pay["paid_at"] < "2026-08-01")]

# У какой доли платежей июля дата «на часах клиента» не совпала с датой в базе?
# Доля — это среднее от колонки из True и False.
other_day = july["local"].dt.date != july["paid_at"].dt.date
share = other_day.ЗАПОЛНИТЕ()

print(f"не в своём дне: {share:.1%} платежей")


In [ ]:
by_zone = other_day.groupby(july["time_zone"]).mean().sort_values()

ax = by_zone.mul(100).round(1).plot.barh(
    figsize=(7, 3.6), color="#6B7280",
    title="Платежи, у которых местная дата не совпала с датой в базе, %")
ax.set_ylabel("");


**Каждый пятый платёж лежит в базе не в том дне, в котором его помнит клиент.**
В Калининграде таких 8,6 %, в Москве 12,4 %, на Камчатке — 47,5 %: почти половина.

Доля равна смещению пояса, делённому на 24 часа: платежи идут ровно круглые сутки,
и «чужими» оказываются те, что попали в последние часы дня по UTC.

**Знать, что существует.** `Series.plot.barh()` — график одной строкой. Не для
отчёта, а чтобы посмотреть самому.


In [ ]:
# А месяц? «Июль по UTC» и «июль на часах клиента» — это одни и те же платежи?
utc_july = (pay["paid_at"] >= "2026-07-01") & (pay["paid_at"] < "2026-08-01")
local_july = (pay["local"] >= "2026-07-01") & (pay["local"] < "2026-08-01")
moved = utc_july != local_july

print(f"выручка июля по UTC:       {pay.loc[utc_july, 'amount'].sum():>12,.0f} ₽".replace(",", " "))
print(f"выручка июля по местному:  {pay.loc[local_july, 'amount'].sum():>12,.0f} ₽".replace(",", " "))
print(f"платежей, которые сменили месяц: {moved.sum()} — это {moved.sum() / utc_july.sum():.2%}")


Вот зачем был запас в один день: вечер 30 июня по UTC — это уже июль для всех,
кто восточнее Гринвича.

**На днях пояс сдвигает каждый пятый платёж. На месяцах — один из ста,** а выручка
июля меняется на 718 ₽ из четырнадцати миллионов: сколько платежей ушло в август,
столько же пришло из июня.

Сегодняшние три числа месячные, поэтому считаем их в UTC и почти ничего не теряем.
А вот дневной алерт вашей системы мониторинга пояс исказит всерьёз.

**Понимать.** Пояс суток — часть определения метрики. В курсе считаем в UTC
и пишем это в подписи к числу.

> **Зачем это в работе.** Обращения в поддержку, сверка с партнёром из другого
> пояса, отчёт «за вчера» в девять утра по Москве, когда на Камчатке уже вечер.


---

## 2. Воронка по каналам

Прежде чем считать деньги — посмотрим, где каналы вообще отличаются друг от друга.

Берём тех, кто зарегистрировался с февраля по июнь 2025 года: у них было больше
года, чтобы пройти все шаги.

> **Воронку считают по тем, у кого было время пройти все шаги.** Клиент, пришедший
> вчера, не «не купил» — он ещё не успел.


In [ ]:
GROUP = ("2025-02-01", "2025-07-01")   # регистрация: февраль–июнь 2025, правый конец не включён

grp = clients[(clients["registered_at"] >= GROUP[0]) & (clients["registered_at"] < GROUP[1])]

pay_g = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    join prime.clients c on c.client_id = s.owner_client_id
    where c.{SEGMENT}
      and c.registered_at >= :a and c.registered_at < :b
      and p.status = 'success'
    order by p.payment_id
""", a=GROUP[0], b=GROUP[1])

print(f"клиентов в группе: {len(grp):>7}")
print(f"их платежей:       {len(pay_g):>7}")


In [ ]:
# У клиента одна подписка или ни одной — это и объявляем в validate.
f = grp.merge(subs[["subscription_id", "owner_client_id", "started_at", "ended_at"]],
              left_on="client_id", right_on="owner_client_id", how="left", validate="1:1")

f["subscribed"] = f["subscription_id"].notna()
lag = f["started_at"] - f["registered_at"]

# «Конверсия в подписку» — за какой срок после регистрации? Назовите окно.
f["subscribed_in_window"] = lag < pd.Timedelta(days=ЗАПОЛНИТЕ)

print(f"подписались в окне: {f['subscribed_in_window'].mean():.1%}")
print(f"подписались вообще: {f['subscribed'].mean():.1%}")


**34 % и 70 % — это одна и та же воронка.** «Конверсия в подписку» без окна — не
число, а половина числа: по нему нельзя сравнить два месяца и нельзя повторить
расчёт.

**Понимать.** Окно пишут рядом с конверсией всегда: «34 % за 30 дней».

> **Зачем это в работе.** «Конверсия упала» — первое, что вам скажут в понедельник.
> Первый встречный вопрос аналитика: «За какое окно?»


In [ ]:
# Оплата засчитывается, пока подписка жива, — договорённость с лекции.
p = pay_g.merge(subs[["subscription_id", "ended_at"]], on="subscription_id", validate="m:1")
p = p[p["ended_at"].isna() | (p["paid_at"] <= p["ended_at"])]

# Номер платежа внутри подписки: первый, второй, третий...
# cumcount считает строки с нуля — внутри чего?
p = p.sort_values(["subscription_id", "paid_at"])
p["k"] = p.groupby(ЗАПОЛНИТЕ).cumcount() + 1

# Одна колонка из другой таблицы — это map, а не merge.
f["n_paid"] = f["subscription_id"].map(p.groupby("subscription_id")["k"].max())

print(f["n_paid"].describe().round(1).to_string())


In [ ]:
for k in (1, 2, 6):
    f[f"paid_{k}"] = f["n_paid"] >= k

steps = ["subscribed", "paid_1", "paid_2", "paid_6"]

# Сумма по колонке из True и False — это «сколько True».
# Что группировка сделает с клиентами, у которых канал не записан?
funnel = f.groupby("acquisition_channel", dropna=ЗАПОЛНИТЕ)[steps].sum()
funnel.insert(0, "registered", f.groupby("acquisition_channel", dropna=False).size())
funnel.index = funnel.index.fillna("без канала")

funnel


In [ ]:
# Конверсия шага — доля от ПРЕДЫДУЩЕГО шага. Сдвигаем таблицу на одну колонку
# и делим. По какой оси сдвигать? Оба варианта отработают без ошибки.
conv = funnel.div(funnel.shift(axis=ЗАПОЛНИТЕ))

conv.iloc[:, 1:].style.format("{:.1%}").background_gradient(cmap="Greens", axis=None)


**Вход у каналов одинаковый.** Подписку оформляют около 70 % в любом канале, первую
оплату делают 98–99 %. На входе каналы неразличимы.

**Расходятся каналы на удержании.** От второй оплаты до шестой у `organic` доходят
90 %, у `performance` — 79 %, у `social` — 74 %.

Значит, каналы различаются не тем, как клиент входит, а тем, как долго он остаётся.
Это уже не воронка — это отток.

**Знать, что существует.** `cumcount()` нумерует строки внутри группы; `.style`
раскрашивает таблицу и подсвечивает, где воронка течёт. `shift(axis=0)` в той же
ячейке не упал бы — он молча поделил бы один канал на другой.


---

## 3. Юнит-экономика: CAC, отток, LTV

Это ядро пары. Три определения с лекции:

- **CAC** — расходы на привлечение, делённые на число привлечённых за тот же период.
- **Отток** — доля подписок, активных на начало периода, которые закончились
  до его конца.
- **LTV ≈ ARPPU / отток** — сколько подписка принесёт за всю жизнь, если отток
  и платёж не изменятся.


In [ ]:
# Расходы на привлечение по регионам не делятся, поэтому CAC считаем по всей базе.
spend = q("select channel, spend_date, amount from prime.marketing_spend")

acquired = q("""
    select c.acquisition_channel as channel,
           count(*) as registered,
           count(s.subscription_id) as subscribed
    from prime.clients c
    left join prime.subscriptions s on s.owner_client_id = c.client_id
    where c.registered_at >= '2025-01-06' and c.registered_at < '2026-08-30'
    group by 1
    order by 1
""")

print("расходы:", spend["spend_date"].min(), "…", spend["spend_date"].max())
acquired


In [ ]:
cac = (spend.groupby("channel", as_index=False)
            .agg(spent=("amount", "sum"))
            .merge(acquired, on="channel", how="left", validate="1:1"))

# На канал потратили десятки миллионов. На кого делим: на всех, кто
# зарегистрировался, или на тех, кто оформил подписку?
# Подсказка: LTV дальше считаем НА ПОДПИСКУ.
cac["cac"] = cac["spent"] / cac[ЗАПОЛНИТЕ]
cac["cac_registered"] = cac["spent"] / cac["registered"]

cac.round(0)


**У CAC и LTV должен быть один и тот же «клиент».** LTV мы считаем на подписку —
значит, и расходы делим на подписавшихся. На зарегистрированных CAC в полтора раза
меньше, и это другой показатель: делить на него LTV подписки нельзя.

Самый дешёвый клиент — у `organic`, 212 ₽. Самый дорогой — у `performance`,
2 347 ₽: в одиннадцать раз дороже.

И оговорка с лекции: у 6 259 клиентов канал не записан. Деньги на них потрачены
и лежат в расходах каналов, а в знаменатель они не попали — CAC каждого канала
завышен примерно на 2 %.


In [ ]:
def churn_rate(subs: pd.DataFrame, start: str, end: str) -> float:
    """Отток за период: доля подписок, активных на start, которые закончились до end."""
    start, end = pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC")
    # Знаменатель. Кого берём: всех, кто был жив в течение периода,
    # или только тех, кто был жив на его начало?
    active = subs[(subs["started_at"] < ЗАПОЛНИТЕ)
                  & (subs["ended_at"].isna() | (subs["ended_at"] >= start))]
    return (active["ended_at"] < end).sum() / len(active)


print(f"отток июля по всем каналам: {churn_rate(subs, '2026-07-01', '2026-08-01'):.2%}")


**На начало периода.** Подписки, оформленные в июле, в знаменатель не входят: иначе
отток зависел бы от того, сколько продали в этом же месяце. Хороший месяц продаж
сам собой «снижал» бы отток.

**Уметь писать.** Маска «активна на дату»: началась раньше **и** не закончилась
или закончилась позже.


In [ ]:
# Клиентов без канала держим отдельной строкой — как договорились на третьем семинаре.
subs["channel"] = subs["acquisition_channel"].fillna("без канала")
pay["channel"] = pay["acquisition_channel"].fillna("без канала")

# Та же функция — для каждого канала отдельно. apply отдаёт ей кусок таблицы.
churn = subs.groupby("channel").apply(ЗАПОЛНИТЕ, "2026-07-01", "2026-08-01")

churn.mul(100).round(2).sort_values()


In [ ]:
july = pay[(pay["paid_at"] >= "2026-07-01") & (pay["paid_at"] < "2026-08-01")]

# ARPPU — выручка на ПЛАТЯЩУЮ подписку. Сколько подписок платило в июле:
# считать строки или считать разные подписки? До запуска скажите, что больше.
by_channel = july.groupby("channel").agg(
    revenue=("amount", "sum"),
    payments=("payment_id", "size"),
    paying=("subscription_id", ЗАПОЛНИТЕ),
)
by_channel["arppu"] = by_channel["revenue"] / by_channel["paying"]

print("платежей на одну платящую подписку:",
      round(by_channel["payments"].sum() / by_channel["paying"].sum(), 3))
by_channel.round(2)


**1,032 платежа на подписку.** Подписка списывается раз в 30 дней, а в июле 31 день:
кто заплатил первого числа, платит ещё раз тридцать первого. Посчитали бы строки —
получили бы на 3 % больше «платящих», чем их есть.

ARPPU у каналов одинаковый, 335–338 ₽: тарифы у всех одни. Значит, всю разницу
в LTV сделает отток.

> К тридцати дням и тридцати одному мы вернёмся на второй паре — там они окажутся
> главным героем.


In [ ]:
unit = pd.DataFrame({
    "churn": churn,
    "arppu": by_channel["arppu"],
    "cac": cac.set_index("channel")["cac"],
})
unit["ltv"] = unit["arppu"] / unit["churn"]
unit["ltv_cac"] = unit["ltv"] / unit["cac"]

# Эмпирическое правило: канал окупается, если LTV хотя бы втрое больше CAC;
# меньше единицы — канал убыточен. pd.cut раскладывает числа по корзинам.
unit["verdict"] = pd.cut(unit["ltv_cac"], bins=[0, 1, ЗАПОЛНИТЕ, np.inf],
                         labels=["убыточен", "на грани", "окупается"])

unit.sort_values("ltv_cac").round({"churn": 4, "arppu": 2, "cac": 0, "ltv": 0, "ltv_cac": 1})


**Что вы ответите маркетингу?** Запишите одним предложением прямо здесь: куда
добавить, что резать — и насколько вы этим числам верите.

*(дважды щёлкните по ячейке и напишите)*

Подсказка для сомнений: посмотрите на LTV/CAC у `organic`. Отток 2,2 % в месяц —
это сколько месяцев жизни подписки? А сколько месяцев истории у нас в базе?


---

## 4. Чего формула не знает

Тот же расчёт, одна правка: отток берём не за июль, а за август. Данные кончаются
29 августа, поэтому окно — с 1-го по 29-е.


In [ ]:
print("последняя отмена в данных:", subs["ended_at"].max())

churn_aug = subs.groupby("channel").apply(churn_rate, "2026-08-01", "2026-08-30")

flip = unit[["churn", "ltv_cac", "verdict"]].copy()
flip["churn_aug"] = churn_aug
flip["ltv_cac_aug"] = unit["arppu"] / churn_aug / unit["cac"]
flip["verdict_aug"] = pd.cut(flip["ltv_cac_aug"], bins=[0, 1, 3, np.inf],
                             labels=["убыточен", "на грани", "окупается"])

flip[["churn", "churn_aug", "ltv_cac", "ltv_cac_aug", "verdict", "verdict_aug"]].sort_values("ltv_cac_aug").round(3)


**`partner_telecom`: отток 3,8 % → 81,5 %.** Канал, который по июльским данным
окупался почти в девять раз, по августовским не возвращает и половины CAC.

22 и 23 августа у него отменили 27 тысяч подписок — тот самый обвал, который
на лекции раскладывали по каналам. За два дня «окупается» превратилось
в «убыточен».

**Понимать.** Формула LTV считает, что отток не изменится. Она ничего не знает
о том, чего ещё не случилось, — и о том, что случилось вчера, тоже, пока вы
не пересчитали.


---

## 5. LTV по данным и срок окупаемости

*Если успеем на паре — иначе дома.*

Формула обещает будущее. А сколько клиенты **уже** принесли? Берём ту же группу,
что в воронке, и складываем её выручку по месяцам жизни клиента.

Здесь считаем на **зарегистрированного** клиента — значит, и CAC берём на
зарегистрированного. Знаменатель снова один и тот же с обеих сторон.


In [ ]:
life = (pay_g
        .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
        .merge(grp[["client_id", "registered_at", "acquisition_channel"]],
               left_on="owner_client_id", right_on="client_id", validate="m:1"))
life["life_month"] = (life["paid_at"] - life["registered_at"]).dt.days // 30 + 1

# Самый молодой клиент группы пришёл 30 июня 2025 и прожил 14 полных месяцев.
# Дальше 14-го кривой верить нельзя: старшие ещё платят, а младшие не дожили.
by_month = life[life["life_month"] <= 14].pivot_table(
    index="life_month", columns="acquisition_channel", values="amount",
    aggfunc="sum", fill_value=0)

# Выручка НАКОПЛЕННЫМ итогом на одного зарегистрированного клиента канала.
curve = by_month.ЗАПОЛНИТЕ() / grp.groupby("acquisition_channel").size()

curve.loc[[1, 3, 6, 12, 14]].round(0)


In [ ]:
cac_reg = cac.set_index("channel")["cac_registered"]

# Срок окупаемости — первый месяц жизни, когда накопленная выручка догнала CAC.
# На колонке из True и False эта функция возвращает ПЕРВЫЙ True.
hit = curve.ge(cac_reg, axis=1)
payback = hit.ЗАПОЛНИТЕ().where(hit.any())

pd.DataFrame({
    "cac_registered": cac_reg.round(0),
    "revenue_14m": curve.loc[14].round(0),
    "ratio": (curve.loc[14] / cac_reg).round(2),
    "payback_month": payback,
}).sort_values("ratio")


In [ ]:
ax = curve.plot(figsize=(9, 4.6), title="Накопленная выручка на зарегистрированного клиента, ₽")
ax.axhline(cac_reg["performance"], color="#C1272D", linestyle="--")
ax.annotate("CAC performance", (1, cac_reg["performance"]), textcoords="offset points",
            xytext=(0, 5), color="#C1272D")
ax.set_xlabel("месяц жизни клиента")
ax.legend(title="канал")
ax.spines[["top", "right"]].set_visible(False);


**`performance` отбивает свой CAC на двенадцатом месяце.** `organic` — на втором,
`crm` — на третьем, `social` — на седьмом.

За 14 месяцев клиент `performance` принёс 1,2 своего CAC. Формула обещала 2,5.
Формула смотрит в будущее, данные — в прошлое; правда между ними, и ближе к данным.

**Понимать.** LTV по данным — нижняя граница: клиенты ещё платят. Срок окупаемости
честнее отношения LTV/CAC: он говорит, сколько месяцев деньги заморожены.

**Знать, что существует.** `idxmax()` на булевой колонке — «где первый True».
Ловушка: если True нет вовсе, она вернёт первую строку, будто канал окупился
в первый же месяц. Поэтому рядом стоит `.where(hit.any())`.

Обратите внимание на деление в первой ячейке: таблица делится на `Series`, и pandas
сам сопоставляет колонки таблицы с метками ряда. Сравнение так не умеет — ему
нужен `ge(..., axis=1)`.


---

## 6. Метрики переезжают в код

Две функции этой пары пригодятся в проекте — `local_time` и `churn_rate`.

**Как перенести к себе.** Откройте `src/prime/metrics.py` в своём `prime-monitor`:
там уже лежит `revenue()` со второго семинара. Допишите сверху `import pandas as pd`,
а в конец файла — обе функции. После этого в любом ноутбуке работает:

```python
from prime.metrics import churn_rate
```

Коммит — в `main`: `git add src/prime/metrics.py && git commit -m "метрики
с семинара 5"`.


---

## Что унести с сегодняшнего дня

1. **Пояс суток — часть определения метрики.** На днях он сдвигает каждый пятый
   платёж, на месяцах — один из ста. Считаем в UTC и пишем это в подписи.
2. **Время с поясом и без — разные типы.** `Timestamp` без пояса падает, строка
   проходит молча.
3. **Конверсия без окна — половина числа.** И считают воронку по тем, у кого было
   время пройти все шаги.
4. **Каналы различаются не входом, а удержанием.**
5. **У CAC и LTV один и тот же «клиент» в знаменателе.**
6. **LTV по формуле — прогноз при неизменном оттоке.** Август он не предвидел.
7. **LTV по данным — нижняя граница,** а срок окупаемости честнее отношения.

### Где вы запустили то, что было на лекции

| Слайд Л5 | Приём | Где |
|---|---|---|
| 5 | время с поясом и без, `TypeError` | раздел 1 |
| 6–7 | `tz_localize`, `tz_convert` | раздел 1 |
| 8–9 | 11 поясов, местная дата платежа | раздел 1 — `local_time()` |
| 11 | какие сутки правильные | раздел 1, «чей июль» |
| 12 | пояс — это имя, а не «+3» | ★5 и ★6 |
| 14–15 | `resample`, `"ME"` | ★1 |
| 16–17 | равные окна, неполный период | раздел 4 — август с 1-го по 29-е |
| 4, 18 | месяц — не единица измерения | раздел 3 (1,032 платежа), ★7 |
| 19–20 | `shift`, накопительные расчёты | раздел 2 (`shift(axis=1)`), раздел 5, ★3 |
| 22–23 | воронка и её окно | раздел 2 |
| 24 | ARPU и знаменатель | раздел 3 — ARPPU |
| 25–26 | отток за период | раздел 3 — `churn_rate()` |
| 27 | LTV: формула и нижняя граница по данным | разделы 3, 4 и 5 |
| 28 | CAC и кто такой «клиент» | раздел 3 |

### К следующей паре

Вторая пара сегодня — **факторный анализ**: в ноябре выручка ПРАЙМ упала, хотя база
росла. Разбираемся, за счёт чего, — и там всплывут те самые 1,032 платежа
на подписку.


---

## Хвост: ваш сегмент

Дальше — работа с открытым концом. Сколько успеете на паре, столько успеете;
остальное дома. Ничего из этого не сдаётся.

У каждого свой сегмент клиентов — тот же, что в ДЗ-2. Условие лежит в вашей строке
`prime.assignments`, в колонке `client_filter`; следующая ячейка достаёт его сама.


In [ ]:
my_filter = q("select client_filter from prime.assignments where login = current_user").iloc[0, 0]

print(my_filter)


In [ ]:
# Задача 1. Платежи июля вашего сегмента: какая доля лежит не в своём дне?
# Сравните с общими 19,5 %. Если разница заметная — посмотрите на пояса сегмента.
mine = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount,
           c.time_zone, c.acquisition_channel
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    join prime.clients c on c.client_id = s.owner_client_id
    where c.{my_filter}
      and p.status = 'success'
      and p.paid_at >= '2026-07-01' and p.paid_at < '2026-08-01'
""")

my_share = None   # ← ваш расчёт

print(len(mine), "платежей; не в своём дне:", my_share)


In [ ]:
# Задача 2. Отток июля по каналам в вашем сегменте — функцией churn_rate.
# Порядок каналов тот же, что на общем срезе?
my_subs = q(f"""
    select s.subscription_id, s.started_at, s.ended_at,
           coalesce(c.acquisition_channel, 'без канала') as channel
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
    where c.{my_filter}
""")

my_churn = None   # ← ваш расчёт

print(my_churn)


### Задача 3. Дно

На лекции было сказано: **для решения нужна маржа, а не выручка.** Подписчик
получает кэшбэк за покупки в сервисах — это деньги, которые ПРАЙМ отдаёт обратно.

Посчитайте LTV по каналам не на выручке, а на марже: ARPPU июля минус кэшбэк июля
на одну платящую подписку. Насколько изменится таблица из раздела 3?

Эта задача намеренно трудная. В ней спрятан вопрос без готового ответа: кэшбэк
начислен в июле за покупку июня — это расход какого месяца?


In [ ]:
# Задача 3. Кэшбэк июля по каналам на общем срезе — готовым запросом.
# Дальше сами: маржа на платящую подписку и LTV на марже.
cashback = q(f"""
    select coalesce(c.acquisition_channel, 'без канала') as channel,
           sum(b.bonus_amount) as cashback
    from prime.cashback b
    join prime.transactions t on t.transaction_id = b.transaction_id
    join prime.clients c on c.client_id = t.client_id
    where c.{SEGMENT}
      and b.accrued_at >= '2026-07-01' and b.accrued_at < '2026-08-01'
    group by 1
""")

ltv_margin = None   # ← ваш расчёт

print(ltv_margin)


---

## Задачи со звёздочкой

Восемь приёмов работы со временем, до которых руки на паре не дошли. Половины
из них не было и на лекции. **Не сдаются и не оцениваются** — но каждый встретится
в проекте или на работе, и лучше узнать его в лицо здесь.

Задачи опираются на таблицы из разделов 1–3: `pay`, `subs`, `clients`. Если
перезапускали ядро — сначала прогоните эти разделы.

Проверка встроена: она знает правильный ответ, но не показывает его. Замените
`None` на своё решение и запустите ячейку целиком. Формат ответа задан в условии —
без него хеш не сойдётся.


In [ ]:
# Это просто запустите
ANSWERS = {
    "★1": "f9808343d6",
    "★2": "4c9bdf897d",
    "★3": "4e07408562",
    "★4": "7902699be4",
    "★5": "e20219c56b",
    "★6": "df3c08c16d",
    "★7": "ebfa18986b",
    "★8": "71cba97a67"
}


In [ ]:
# Это тоже
import hashlib

import numpy as np
import pandas as pd


def _norm(x):
    if isinstance(x, bool):
        return repr(x)
    if isinstance(x, float):
        return f"{x:.2f}"
    if isinstance(x, int):
        return str(x)
    if isinstance(x, dict):
        return "{" + ",".join(f"{_norm(k)}:{_norm(v)}" for k, v in sorted(x.items(), key=lambda kv: _norm(kv[0]))) + "}"
    if isinstance(x, (set, frozenset)):
        return "{" + ",".join(sorted(_norm(v) for v in x)) + "}"
    if isinstance(x, (list, tuple)):
        return "[" + ",".join(_norm(v) for v in x) + "]"
    if isinstance(x, np.integer):
        return str(int(x))
    if isinstance(x, np.bool_):
        return repr(bool(x))
    if isinstance(x, pd.Timestamp):
        return x.isoformat()
    if isinstance(x, pd.Timedelta):
        return str(int(x.total_seconds()))
    if isinstance(x, pd.Index):
        return _norm(list(x))
    if isinstance(x, pd.Series):
        return "{" + ",".join(f"{_norm(k)}:{_norm(v)}" for k, v in sorted(x.items(), key=lambda kv: _norm(kv[0]))) + "}"
    if isinstance(x, pd.DataFrame):
        d = x if isinstance(x.index, pd.RangeIndex) else x.reset_index()
        cols = sorted(map(str, d.columns))
        rows = sorted(tuple(_norm(r[c]) for c in cols) for _, r in d.iterrows())
        return "[" + ",".join("[" + ",".join(r) + "]" for r in rows) + "]"
    return repr(x)


def _show(x) -> str:
    """Короткий показ ответа: кадр на сто тысяч строк в вывод не печатаем."""
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return f"{type(x).__name__} {x.shape}\n{x.head().to_string()}"
    return repr(x)


def check(task: str, got) -> None:
    """Сверить ответ с эталоном. Эталоны лежат хешами — подсмотреть нельзя."""
    if got is None:
        print(f"{task}: не решено")
        return
    digest = hashlib.sha256(_norm(got).encode()).hexdigest()[:10]
    if not ANSWERS:
        print(f"HASH:{task}={digest}")
    elif digest == ANSWERS.get(task):
        print(f"{task}: верно")
    else:
        print(f"{task}: не сходится, получилось {_show(got)}")


print("проверка готова")


### ★ 1. Прирост к прошлому месяцу одним вызовом

Слайды 14–15 и 19. Посчитайте, сколько подписок среза оформляли в каждом месяце
2025 года: `resample("ME")` по `started_at`. Прирост к предыдущему месяцу —
не `shift` и деление руками, а `pct_change()`.

**Ответ:** строка вида `"2025-05"` — месяц с самым большим приростом среди
февраля–декабря 2025 года.


In [ ]:
started = subs.set_index("started_at").sort_index()

answer = None   # ← ваш расчёт

check("★1", answer)


### ★ 2. День недели

Слайд 4. У колонки со временем есть `dt.day_name()` — название дня недели.

Сколько платежей июля (по UTC, таблица `july`) пришлось на пятницы? И сколько
пятниц было в июле 2026 года?

**Ответ:** список из двух целых — `[платежей в пятницы, пятниц в июле]`.

Сравните со вторниками — и вспомните слайд «месяц — не единица измерения».


In [ ]:
answer = None   # ← ваш расчёт

check("★2", answer)


### ★ 3. Дни-рекорды

Слайды 19–20. Соберите дневную выручку июля (по UTC). День-рекорд — день, в который
выручка **строго больше**, чем в любой из предыдущих дней месяца. Первое июля
рекордом не считаем: сравнивать ему не с чем.

Подсказка: `cummax()` — наибольшее из всех значений до текущего включительно.
Чтобы сравнить день с **предыдущими**, ряд надо сдвинуть.

**Ответ:** целое число — сколько в июле дней-рекордов.


In [ ]:
daily = july.groupby(july["paid_at"].dt.date)["amount"].sum()

answer = None   # ← ваш расчёт

check("★3", answer)


### ★ 4. Соединение «по ближайшему раньше»

Этого на лекции не было. `pd.merge_asof` соединяет не по равенству ключей, а по
**ближайшему предыдущему** значению — и умеет ограничить, насколько далеко искать.

Вопрос поддержки: сколько подписок среза, отменённых в июле 2026 года, отменили
в течение 7 дней после обращения владельца в поддержку?

Для каждой отмены найдите последнее обращение её владельца не раньше чем за 7 дней
до отмены: `left_on`, `right_on`, `left_by`, `right_by`, `tolerance`. Обе таблицы
уже отсортированы по времени — без этого `merge_asof` падает.

**Ответ:** целое число — у скольких отмен такое обращение нашлось.


In [ ]:
tickets = q(f"""
    select t.client_id, t.created_at
    from prime.support_tickets t
    join prime.clients c on c.client_id = t.client_id
    where c.{SEGMENT}
      and t.created_at >= '2026-06-20' and t.created_at < '2026-08-01'
    order by t.created_at
""")
ended = (subs[(subs["ended_at"] >= "2026-07-01") & (subs["ended_at"] < "2026-08-01")]
         .sort_values("ended_at"))

answer = None   # ← ваш расчёт

check("★4", answer)


### ★ 5. Сутки, в которых не 24 часа

Слайд 12. База не нужна. В Берлине часы переводят дважды в год: в 2025 году —
30 марта и 26 октября.

Постройте почасовой ряд за каждые из этих суток: `pd.date_range` с `freq="h"`,
`tz="Europe/Berlin"` и `inclusive="left"` (от полуночи до следующей полуночи).

**Ответ:** список из двух целых — `[часов в сутках 30 марта, часов в сутках 26 октября]`.


In [ ]:
answer = None   # ← ваш расчёт

check("★5", answer)


### ★ 6. Местное время без цикла

Слайд 12. Функция `local_time` идёт циклом по поясам. В России часы не переводят
с 2014 года, поэтому у каждого нашего пояса смещение от UTC постоянно — и местное
время можно получить одним сложением.

1. Постройте словарь «пояс → смещение»:
   `pd.Timestamp("2026-07-01", tz=...).utcoffset()` для каждого пояса из `pay`.
2. Убедитесь `assert`-ом, что в январе смещения те же. В Берлине эта проверка упала бы.
3. Местное время — `paid_at` без пояса плюс смещение: `map` по колонке `time_zone`.
4. Убедитесь, что результат совпал с колонкой `local`.

**Ответ:** целое число — сколько платежей из `pay` прошло ночью, с 00:00 до 06:00
по часам клиента.


In [ ]:
answer = None   # ← ваш расчёт

check("★6", answer)


### ★ 7. Сколько в день

Слайд 18. Сколько подписок среза оформляли **в день** в феврале и в марте 2025 года?
Число дней в месяце знает сам индекс: `days_in_month`.

**Ответ:** список из двух чисел с двумя знаками — `[в день в феврале, в день в марте]`.


In [ ]:
started = subs.set_index("started_at").sort_index()

answer = None   # ← ваш расчёт

check("★7", answer)


### ★ 8. Чей июль — в рублях

Раздел 1. На сколько рублей выручка июля по UTC отличается от выручки июля
«на часах клиента»?

**Ответ:** одно число с двумя знаками — `round(по UTC - по местному, 2)`.


In [ ]:
answer = None   # ← ваш расчёт

check("★8", answer)
